
###### Author: Archie Hulse


#                          Critical Evaluation of Preprocessing, Language Models and Classification in NLP - SMS Spam Classification

------------------------------
#### <u>Contents:</u>
###### *(Hyperlinks to sections of this report)*
[Go to Section 1](#1) - Imports, Setup, Dataset Loading and Inspection

[Go to Section 2](#2) - Preprocessing

[Go to Section 3](#3) - Language Modelling (Word2Vec)

[Go to Section 4](#4) - Classification

[Go to Section 5](#5) - Error Analysis

[Go to Section 6](#6) - References

------------------------------

###### 1
## 1. - <u>Imports, Setup, Dataset Loading and Inspection:<u/>

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from collections import Counter
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

In [ ]:
#load the data:
data_candidates = [Path("data/SMSSpamCollection"), Path("../data/SMSSpamCollection")]
file_path = next((p for p in data_candidates if p.exists()), data_candidates[0])

df = pd.read_csv(file_path, sep='\t', header=None, names=['label', 'message'])   # tab-separated, no header 

In [ ]:
print(df.head())

# count each label
print("\nLabel counts:")
print(df['label'].value_counts())

#summary stats of message length
df['message_length'] = df['message'].apply(len)
print("\nMessage length summary:")
print(df['message_length'].describe())

------------------------------------
###### 2
## 2. - <u>Preprocessing:</u>

In [ ]:
# will carry out 2 piplines of preprocessing to invetigate later effects of this on vocabulary size and type-token ratio:

# had to force download to local folder as was breaking code if not
nltk.download('punkt_tab', download_dir='nltk_data')
nltk.download('stopwords', download_dir='nltk_data')
nltk.download('wordnet', download_dir='nltk_data')

# Make NLTK searches this folder
nltk.data.path.append('nltk_data')

# making lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))


# Pipeline 1: minimal preprocessing
def preprocess_min(text):
    text = str(text)           # ensure it's a string
    text = text.lower()  #lowercase
    tokens = word_tokenize(text) #tokenize
    return tokens

# Pipeline 2: maximum preprocessing
def preprocess_max(text):
    text = str(text)
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha()]          #removes punctuation & numbers
    tokens = [t for t in tokens if t not in stop_words]  # removes the stopwords
    tokens = [lemmatizer.lemmatize(t) for t in tokens]   #lemmatize
    return tokens

# apply to the SMS dataset
df['tokens_minimal'] = df['message'].apply(preprocess_min)
df['tokens_max'] = df['message'].apply(preprocess_max)


print(df[['message', 'tokens_minimal', 'tokens_max']].head())

In [ ]:
#compare preprocessing outputs for Vocabulary & Type-Token Ratio:

def vocab_stats(tokens_list):
    all_tokens = [token for tokens in tokens_list for token in tokens]
    vocab_size = len(set(all_tokens))
    total_tokens = len(all_tokens)
    ttr = vocab_size / total_tokens
    top10 = Counter(all_tokens).most_common(10)
    return vocab_size, total_tokens, ttr, top10

#Min preprocessing stats
vocab_min, total_min, ttr_min, top10_min = vocab_stats(df['tokens_minimal'])
print("Minimal Preprocessing:")
print(f"Vocabulary size: {vocab_min}, Total tokens: {total_min}, TTR: {ttr_min:.3f}")
print(f"Top 10 tokens: {top10_min}\n")

#Max preprocessing stats
vocab_aggr, total_aggr, ttr_aggr, top10_aggr = vocab_stats(df['tokens_max'])
print("Maximum Preprocessing:")
print(f"Vocabulary size: {vocab_aggr}, Total tokens: {total_aggr}, TTR: {ttr_aggr:.3f}")
print(f"Top 10 tokens: {top10_aggr}")


----------------------------------------
###### 3
## 3. - <u>Language Modelling (Word2Vec):</u>

In [ ]:
# training a model with Word2Vec embeddings on maximum preprocessed tokens
w2vec_model = Word2Vec(
    sentences=df['tokens_max'],
    vector_size=50,   
    window=5,         #context window
    min_count=5,      # ignores very rare words
    workers=2,        # only small number of threads
    seed=42
)

# Check vector for a word
print("Vector for word 'free':\n", w2vec_model.wv['free'])

# gets the nearest neighbors for words I think are accosiated with SMS Spam:
sample_words = ['free', 'call', 'win', 'money', 'urgent']
for word in sample_words:
    if word in w2vec_model.wv.key_to_index:
        neighbors = w2vec_model.wv.most_similar(word, topn=5)
        print(f"\nTop neighbors for '{word}': {neighbors}")


In [ ]:
for word in ['free', 'win', 'ok', 'yes', 'call']:
    if word in w2vec_model.wv.key_to_index:
        print(f"{word}: {w2vec_model.wv.most_similar(word, topn=5)}")

In [ ]:
#pca plot to show the clusters of key spam words with their nearest neigbours (nns):
# the spam words
keywords = ['free', 'win', 'money', 'urgent', 'prize']

#color list for each word and their nns.
colors = ['red', 'blue', 'green', 'purple', 'orange']

words = []
vectors = []
word_colors = []
is_keyword = []   #tracks if word is a keyword to bolden later

# collects embeddings
for idx, word in enumerate(keywords):
    if word in w2vec_model.wv.key_to_index:
        #keyword itself
        words.append(word)
        vectors.append(w2vec_model.wv[word])
        word_colors.append(colors[idx])
        is_keyword.append(True)   # keyword

        # nearest neighbours
        neighbors = [w for w, _ in w2vec_model.wv.most_similar(word, topn=5)]
        for n in neighbors:
            words.append(n)
            vectors.append(w2vec_model.wv[n])
            word_colors.append(colors[idx])
            is_keyword.append(False)  # neighbour

# PCA to 2D (as was vector 50)
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(vectors)

#The plot
plt.figure(figsize=(9,7))

for i, word in enumerate(words):
    # Scatter plot
    plt.scatter(
        coords[i,0],
        coords[i,1],
        color=word_colors[i],
        edgecolor='black' if is_keyword[i] else None,  # outline keywords
        s=90 if is_keyword[i] else 50                  # bigger marker for keywords
    )

    # text labels
    plt.text(
        coords[i,0] + 0.02,
        coords[i,1] + 0.02,
        word,
        fontsize=11 if is_keyword[i] else 9,
        fontweight='bold' if is_keyword[i] else 'normal'
    )

plt.title("Word2Vec PCA: Spam Keywords and Nearest Neighbours", pad=20)
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.show()

--------------------------------
###### 4
## 4. - <u>Classification Models:</u>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
# join tokens back into strings for CountVectorizer
df['text_for_model'] = df['tokens_max'].apply(lambda x: ' '.join(x))

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    df['text_for_model'], df['label'], test_size=0.2, random_state=42, stratify=df['label'])

#convert text to BoW
vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
# Naive Bayes model (NB)
nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)
y_pred_nb = nb_model.predict(X_test_vec)

# Logistic Regression model (LR)
lr_model = LogisticRegression(max_iter=200)
lr_model.fit(X_train_vec, y_train)
y_pred_lr = lr_model.predict(X_test_vec)

In [ ]:
#print the model conf. mat. and eval. results:

def print_metrics(y_true, y_pred, model_name):
    print(f"--- {model_name} ---")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, pos_label='spam'))
    print("Recall:", recall_score(y_true, y_pred, pos_label='spam'))
    print("F1:", f1_score(y_true, y_pred, pos_label='spam'))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
    print("\n")

print_metrics(y_test, y_pred_nb, "Naive Bayes")
print_metrics(y_test, y_pred_lr, "Logistic Regression")

---------------------------------------
###### 5
## 5. - <u>Error Analysis:</u>

In [ ]:
import seaborn as sns
from collections import Counter

In [ ]:
# combine predictions with original data
X_test_df = X_test.to_frame()      #origional message sent
X_test_df['true_label'] = y_test   #the true label of spam or ham
X_test_df['pred_lr'] = y_pred_lr   #logistic reg. pred.
X_test_df['pred_nb'] = y_pred_nb   #nive bayes pred.

# filter for Logistic Regression (spam missed or ham incorrectly flagged)
misclassified_lr = X_test_df[X_test_df['true_label'] != X_test_df['pred_lr']]

print(misclassified_lr.head(20))

In [ ]:
# filter for naive bayes (spam missed or ham incorrectly flagged)
misclassified_nb = X_test_df[X_test_df['true_label'] != X_test_df['pred_nb']]

print(misclassified_nb.head(10))

In [ ]:
cm = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['ham','spam'], yticklabels=['ham','spam'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Logistic Regression Confusion Matrix')
plt.show()

In [ ]:
cm = confusion_matrix(y_test, y_pred_nb)

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['ham', 'spam'],
    yticklabels=['ham', 'spam']
)

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Naive Bayes Confusion Matrix')
plt.show()

In [ ]:
#this was a blocker but managed to figure it out in the end. kept getting key error 'tokens_max'

#makes a copy of the test set including tokens and true labels
X_test_df = X_test.to_frame().copy()
X_test_df['true_label'] = y_test
X_test_df['pred_lr'] = y_pred_lr
X_test_df['pred_nb'] = y_pred_nb

#adding only the tokens_max column from the original df
#indices match the X_test indices
X_test_df['tokens_max'] = df.loc[X_test.index, 'tokens_max']

# False negatives for Naive Bayes (spam predicted as ham)
false_negatives_nb = X_test_df[(X_test_df['true_label']=='spam') & (X_test_df['pred_nb']=='ham')]

# gets a list of all words in the false negative predictions
all_tokens_fn_nb = [t for tokens in false_negatives_nb['tokens_max'] for t in tokens]

# counts the top 10 tokens
from collections import Counter
top_tokens_fn_nb = Counter(all_tokens_fn_nb).most_common(10)

print("Top tokens in false negatives (Naive Bayes):", top_tokens_fn_nb)

###### 6
## 6. - <u>References:</u>

Almeida, T. & Hidalgo, J. (2011). SMS Spam Collection [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CC84 

Alpaydin, Ethem, and Francis Bach. *Introduction to Machine Learning*. Third edition. Cambridge: MIT Press, 2014. (chapter 8.2.3) Print.

Antić, Z. Python natural language processing cookbook: over 50 recipes to understand, analyze, and generate text for implementing language processing tasks. (Birmingham: Packt Publishing Ltd, 2021).

Trizkynoviandy. (2025). SMS Spam Classifier [GitHub repository]. https://github.com/trizkynoviandy/sms-spam-classifier 